# Visualise Ancestry 

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.colors as mcolors
import numpy as np
import os
from typing import Optional, List, Dict, Tuple
from matplotlib.lines import Line2D
import matplotlib
from matplotlib.transforms import blended_transform_factory
# Removed seaborn as box plot is removed

# --- Existing Helper Functions (unchanged) ---
def get_unique_ancestries(locus_data_df):
    ancestries = set()
    for chromatid in ['chromatid1', 'chromatid2']:
        pop_col = f'ancestry_{chromatid}_pop'
        founder_col = f'ancestry_{chromatid}_founder_id'
        filtered = locus_data_df[[pop_col, founder_col]].dropna()
        for _, row in filtered.iterrows():
            ancestries.add((row[pop_col], int(row[founder_col])))
    return ancestries

def generate_color_map_for_ancestries(unique_ancestries):
    cmap = matplotlib.colormaps['tab20'].resampled(len(unique_ancestries))
    color_map = {}
    for i, anc in enumerate(sorted(unique_ancestries)):
        rgba = cmap(i)
        color_map[anc] = mcolors.to_hex(rgba)
    return color_map

def consolidate_ancestry_blocks(locus_data: pd.DataFrame, chromatid_label: str):
    blocks = []
    current_block_ancestry = None
    current_block_start_local_idx = 0

    pop_col = f'ancestry_{chromatid_label}_pop'
    founder_col = f'ancestry_{chromatid_label}_founder_id'

    if locus_data.empty:
        return []

    for i, row in locus_data.iterrows():
        anc_pop = row[pop_col]
        founder_id = row[founder_col]

        if pd.isna(anc_pop) or pd.isna(founder_id):
            current_ancestry = (None, None)
        else:
            current_ancestry = (anc_pop, int(founder_id))

        if current_ancestry != current_block_ancestry:
            if current_block_ancestry is not None:
                blocks.append({
                    'start_idx': current_block_start_local_idx,
                    'end_idx': i - 1,
                    'ancestry': current_block_ancestry
                })

            current_block_ancestry = current_ancestry
            current_block_start_local_idx = i

    # Append the last block after the loop
    if current_block_ancestry is not None:
        blocks.append({
            'start_idx': current_block_start_local_idx,
            'end_idx': len(locus_data) - 1,
            'ancestry': current_block_ancestry
        })

    return blocks

def plot_diploid_chromosome_ancestry(
    individual_id: int,
    diploid_chr_id: int,
    generation_label: str,
    locus_data_df: pd.DataFrame,
    marker_height: float = 0.4, # Height of each chromatid bar
    spacing: float = 0.2,      # Vertical spacing between chromatids
    save_filename: Optional[str] = None,
    locus_width_unit: float = 0.1 # Width in inches for each locus to control overall plot width
):
    target_data = locus_data_df[
        (locus_data_df['individual_id'] == individual_id) &
        (locus_data_df['diploid_chr_id'] == diploid_chr_id) &
        (locus_data_df['generation'] == generation_label)
    ].sort_values(by='locus_position').reset_index(drop=True)

    if target_data.empty:
        print(f"No data found for Individual ID {individual_id}, Chromosome {diploid_chr_id} in generation {generation_label}.")
        return

    num_loci = len(target_data)

    unique_ancestries = get_unique_ancestries(locus_data_df)
    ancestry_colors = generate_color_map_for_ancestries(unique_ancestries)

    if (None, None) not in ancestry_colors:
        ancestry_colors[(None, None)] = 'gray'

    # Calculate figure width based on total number of loci
    fig_width = max(10, num_loci * locus_width_unit)
    fig_height = 4.0 # Fixed height, as everything is on one row now

    # Create a single subplot
    fig, ax = plt.subplots(1, 1, figsize=(fig_width, fig_height))

    legend_elements = []
    used_ancestries = set()

    global_locus_start = target_data['locus_position'].iloc[0] if not target_data.empty else 0

    # Define vertical positions for chromatids
    chromatid_y_positions = {
        'chromatid1': marker_height + spacing,
        'chromatid2': 0.0
    }
    y_min_limit = -0.1
    y_max_limit = marker_height + spacing + marker_height + 0.1

    ax.set_title(f"Individual {individual_id}, Chromosome {diploid_chr_id} ({generation_label})\nAncestral Origin", fontsize=14)

    ax.set_ylabel("") # No general Y-axis label needed
    ax.set_yticks([]) # Hide default y-ticks
    ax.set_yticklabels([]) # Hide default y-tick labels

    # X-axis setup for the *entire* chromosome
    tick_interval = max(1, num_loci // 20)
    ticks = np.arange(0, num_loci, tick_interval)
    labels = ticks + global_locus_start

    ax.set_xticks(ticks)
    ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.set_xlabel("Locus Position", fontsize=12)

    ax.set_xlim(-0.5, num_loci - 0.5)
    ax.set_ylim(y_min_limit, y_max_limit)
    ax.set_aspect('auto', adjustable='box')


    # Draw horizontal lines for chromatids (these will be the base lines)
    for chromatid_label_key, y_base in chromatid_y_positions.items():
        ax.plot([-0.5, num_loci - 0.5], [y_base, y_base], color='black', linewidth=1.5, zorder=1)
        ax.plot([-0.5, num_loci - 0.5], [y_base + marker_height, y_base + marker_height], color='black', linewidth=1.5, zorder=1)

        # Labels for chromatids
        transform_x_axes_y_data = blended_transform_factory(ax.transAxes, ax.transData)
        chromatid_display_name = 'Chromatid 1' if chromatid_label_key == 'chromatid1' else 'Chromatid 2'
        ax.text(-0.02, y_base + marker_height/2, chromatid_display_name, va='center', ha='right', fontsize=10, transform=transform_x_axes_y_data)


    # Plot ancestry blocks for each chromatid
    for chromatid_label, y_bottom_pos in chromatid_y_positions.items():
        blocks = consolidate_ancestry_blocks(target_data, chromatid_label)

        for block in blocks:
            block_start_local_idx = block['start_idx']
            block_end_local_idx = block['end_idx']
            ancestry = block['ancestry']

            if ancestry == (None, None):
                display_label = "Missing Data"
                color = ancestry_colors.get((None, None), 'gray')
            else:
                anc_pop, founder_id = ancestry
                display_label = f'{anc_pop}_{founder_id}'
                color = ancestry_colors.get(ancestry, 'gray')

            rect_width = (block_end_local_idx - block_start_local_idx + 1)
            rect_x = block_start_local_idx - 0.5

            rect = patches.Rectangle(
                (rect_x, y_bottom_pos),
                rect_width,
                marker_height,
                facecolor=color,
                edgecolor='none',
                linewidth=0,
                zorder=2
            )
            ax.add_patch(rect)

            if ancestry != (None, None):
                text_x_pos = block_start_local_idx + (block_end_local_idx - block_start_local_idx + 1) / 2 - 0.5
                try:
                    rgb = mcolors.to_rgb(color)
                    text_color = 'black' if np.mean(rgb) > 0.5 else 'white'
                except Exception:
                    text_color = 'black'

                ax.text(
                    text_x_pos,
                    y_bottom_pos + marker_height / 2,
                    str(founder_id),
                    ha='center',
                    va='center',
                    fontsize=7,
                    color=text_color,
                    zorder=3
                )

            if display_label not in used_ancestries:
                legend_elements.append(Line2D([0], [0], marker='s', color='w', label=display_label,
                                              markerfacecolor=color, markersize=8))
                used_ancestries.add(display_label)

    if legend_elements:
        legend_elements.sort(key=lambda x: x.get_label())

        fig.legend(
            handles=legend_elements,
            title="Ancestral Origin",
            loc='center right',
            bbox_to_anchor=(0.98, 0.5),
            borderaxespad=0.0,
            frameon=False,
            fontsize=9,
            title_fontsize=10
        )

    fig.subplots_adjust(left=0.05, right=0.75, top=0.85, bottom=0.15)

    if save_filename:
        plt.savefig(save_filename, bbox_inches='tight', dpi=300)
        print(f"Chromosome plot saved to: {save_filename}")
        plt.close(fig)
    else:
        plt.show()
        return fig

def generate_ancestry_block_summary_df(locus_data: pd.DataFrame) -> pd.DataFrame:
    summary_data = []

    unique_ids_chrs_gens = locus_data[['individual_id', 'diploid_chr_id', 'generation']].drop_duplicates().copy()

    for _, row_id_chr_gen in unique_ids_chrs_gens.iterrows():
        ind_id = row_id_chr_gen['individual_id']
        chr_id = row_id_chr_gen['diploid_chr_id']
        gen_label = row_id_chr_gen['generation']

        target_data = locus_data[
            (locus_data['individual_id'] == ind_id) &
            (locus_data['diploid_chr_id'] == chr_id) &
            (locus_data['generation'] == gen_label)
        ].sort_values(by='locus_position').reset_index(drop=True)

        if target_data.empty:
            continue

        blocks1 = consolidate_ancestry_blocks(target_data, 'chromatid1')
        blocks2 = consolidate_ancestry_blocks(target_data, 'chromatid2')

        summary_data.append({
            'individual_id': ind_id,
            'diploid_chr_id': chr_id,
            'generation': gen_label,
            'num_blocks_chromatid1': len(blocks1),
            'num_blocks_chromatid2': len(blocks2)
        })

    return pd.DataFrame(summary_data)

# Custom sort key for generations (defined once, used by both plotting functions)
def get_generation_sort_key(gen_label):
    if isinstance(gen_label, str):
        if gen_label.startswith('F'):
            try:
                return (0, int(gen_label[1:]), '')
            except ValueError:
                return (0, 0, gen_label)
        elif gen_label.startswith('BC'):
            try:
                num_part_str = ''.join(filter(str.isdigit, gen_label))
                num_part = int(num_part_str) if num_part_str else 0
                alpha_suffix = ''.join(filter(str.isalpha, gen_label.replace('BC', '')))
                return (1, num_part, alpha_suffix)
            except ValueError:
                return (1, 0, gen_label)
    return (2, 0, str(gen_label))


def plot_blocks_per_generation(
    ancestry_summary_df: pd.DataFrame,
    save_filename: Optional[str] = None
):
    """
    Plots the average number of ancestry blocks per chromatid against generation time.
    """
    if ancestry_summary_df.empty:
        print("Ancestry summary DataFrame is empty. Cannot generate blocks per generation plot.")
        return

    summary_df_copy = ancestry_summary_df.copy()
    summary_df_copy['total_blocks'] = summary_df_copy['num_blocks_chromatid1'] + summary_df_copy['num_blocks_chromatid2']
    summary_df_copy['blocks_per_chromatid'] = summary_df_copy['total_blocks'] / 2

    summary_df_copy['generation_sort_key'] = summary_df_copy['generation'].apply(get_generation_sort_key)
    blocks_per_gen = summary_df_copy.groupby('generation_sort_key')['blocks_per_chromatid'].mean().reset_index()

    generation_labels_map = summary_df_copy[['generation_sort_key', 'generation']].drop_duplicates()
    blocks_per_gen = pd.merge(blocks_per_gen, generation_labels_map, on='generation_sort_key', how='left')
    blocks_per_gen = blocks_per_gen.sort_values(by='generation_sort_key')

    if blocks_per_gen.empty:
        print("No aggregated data to plot for blocks per generation.")
        return

    plt.figure(figsize=(10, 6))
    plt.plot(blocks_per_gen['generation'], blocks_per_gen['blocks_per_chromatid'], marker='o', linestyle='-', color='skyblue')

    plt.title('Mean Number of Ancestry Blocks Per Chromatid Over Generations', fontsize=16)
    plt.xlabel('Generation', fontsize=12)
    plt.ylabel('Mean Number of Blocks', fontsize=12) # Changed label to be precise
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()

    if save_filename:
        plt.savefig(save_filename, bbox_inches='tight', dpi=300)
        print(f"Mean Blocks per generation plot saved to: {save_filename}")
        plt.close()
    else:
        plt.show()

# --- NEW FUNCTION: Plot Median and Percentile Blocks Per Generation (Line graph with fill) ---
def plot_median_and_percentile_blocks_per_generation(
    ancestry_summary_df: pd.DataFrame,
    save_filename: Optional[str] = None
):
    """
    Plots the median number of ancestry blocks per chromatid, with a shaded
    region representing the 25th to 75th percentile range (IQR) for individual
    chromosomes within each generation.
    """
    if ancestry_summary_df.empty:
        print("Ancestry summary DataFrame is empty. Cannot generate median block distribution plot.")
        return

    plot_df = ancestry_summary_df.copy()
    plot_df['blocks_per_chromatid'] = (plot_df['num_blocks_chromatid1'] + plot_df['num_blocks_chromatid2']) / 2

    plot_df['generation_sort_key'] = plot_df['generation'].apply(get_generation_sort_key)
    # Sort and get unique generations for consistent plotting order
    sorted_generations_df = plot_df[['generation_sort_key', 'generation']].drop_duplicates().sort_values('generation_sort_key')
    sorted_generations = sorted_generations_df['generation'].tolist()

    # Calculate median, 25th, and 75th percentiles per generation
    grouped_stats = plot_df.groupby('generation_sort_key')['blocks_per_chromatid'].agg(
        median='median',
        q25=lambda x: x.quantile(0.25),
        q75=lambda x: x.quantile(0.75)
    ).reset_index()

    # Merge original generation labels back for plotting
    grouped_stats = pd.merge(grouped_stats, sorted_generations_df, on='generation_sort_key', how='left')
    grouped_stats = grouped_stats.sort_values(by='generation_sort_key')


    plt.figure(figsize=(10, 6))

    # Plot the shaded region for IQR
    plt.fill_between(
        grouped_stats['generation'],
        grouped_stats['q25'],
        grouped_stats['q75'],
        color='lightcoral', # Light color for the shaded area
        alpha=0.4,
        label='25th-75th Percentile Range'
    )

    # Plot the median line
    plt.plot(
        grouped_stats['generation'],
        grouped_stats['median'],
        marker='o',
        linestyle='-',
        color='firebrick', # A darker color for the median line
        label='Median Number of Blocks'
    )


    plt.title('Median and Interquartile Range of Ancestry Blocks Per Chromatid', fontsize=16)
    plt.xlabel('Generation', fontsize=12)
    plt.ylabel('Number of Blocks Per Chromatid', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.xticks(rotation=45, ha='right')
    plt.legend(fontsize=10)
    plt.tight_layout()

    if save_filename:
        plt.savefig(save_filename, bbox_inches='tight', dpi=300)
        print(f"Median and Percentile Blocks per generation plot saved to: {save_filename}")
        plt.close()
    else:
        plt.show()

# --- Main execution block ---
if __name__ == "__main__":
    print("\nStarting Chromosome Ancestry Analysis: Visualization, Summary, and Block Count Plots")

    # Load the locus-level genotype data with ancestry info
    locus_level_df = pd.read_csv('output_data/dataframes/locus_level_genotypes_with_ancestry.csv')

    # Example 1: List IDs in a specific generation
    generation_to_check = 'F3'
    print(f"\n--- Individual IDs in Generation '{generation_to_check}' ---")
    ids_in_generation = locus_level_df.loc[
        locus_level_df['generation'] == generation_to_check,
        'individual_id'
    ].unique().tolist()

    if ids_in_generation:
        print(f"Total individuals in '{generation_to_check}': {len(ids_in_generation)}")
        print("First 10 IDs (if available):", ids_in_generation[:10])
        print("Last 10 IDs (if available):", ids_in_generation[-10:])
    else:
        print(f"No individuals found for generation '{generation_to_check}'.")

    # --- Generate the supplementary DataFrame ---
    print("\nGenerating Ancestry Block Summary DataFrame...")
    ancestry_summary_df = generate_ancestry_block_summary_df(locus_level_df.copy())

    # Save the summary DataFrame
    summary_output_path = 'output_data/dataframes/ancestry_block_summary.csv'
    os.makedirs(os.path.dirname(summary_output_path), exist_ok=True)
    ancestry_summary_df.to_csv(summary_output_path, index=False)
    print(f"Ancestry block summary saved to: {summary_output_path}")
    print("\nFirst 5 rows of the Ancestry Block Summary (including new block counts):")
    print(ancestry_summary_df.head())
    print("\n--- End of Ancestry Block Summary ---")

    plots_directory = 'output_data/plots'
    os.makedirs(plots_directory, exist_ok=True)

    # --- Plotting Mean Blocks Per Generation (your existing line graph) ---
    blocks_per_gen_plot_path = os.path.join(plots_directory, "mean_ancestry_blocks_per_generation.png")
    print(f"\nPlotting mean number of ancestry blocks per generation to: {blocks_per_gen_plot_path}")
    plot_blocks_per_generation(ancestry_summary_df, save_filename=blocks_per_gen_plot_path)

    # --- NEW PLOT: Median and Percentile Blocks Per Generation ---
    median_block_dist_plot_path = os.path.join(plots_directory, "median_ancestry_blocks_per_generation_with_iqr.png")
    print(f"\nPlotting median and percentile ancestry blocks per generation to: {median_block_dist_plot_path}")
    plot_median_and_percentile_blocks_per_generation(ancestry_summary_df, save_filename=median_block_dist_plot_path)

    # --- Plotting a SPECIFIC individual (your existing detailed plot) ---
    specific_id_to_plot = 41
    specific_gen_for_id = 'F3'
    specific_chr_to_plot = 2

    if not locus_level_df[
        (locus_level_df['individual_id'] == specific_id_to_plot) &
        (locus_level_df['generation'] == specific_gen_for_id) &
        (locus_level_df['diploid_chr_id'] == specific_chr_to_plot)
    ].empty:

        plot_save_path_specific_ind = os.path.join(
            plots_directory,
            f"specific_ind_{specific_id_to_plot}_gen_{specific_gen_for_id}_chr_{specific_chr_to_plot}_ancestry_merged.png"
        )

        print(f"\nPlotting specific individual (ID: {specific_id_to_plot}) from generation '{specific_gen_for_id}', Chromosome {specific_chr_to_plot} with merged blocks.")
        plot_diploid_chromosome_ancestry(
            individual_id=specific_id_to_plot,
            diploid_chr_id=specific_chr_to_plot,
            generation_label=specific_gen_for_id,
            locus_data_df=locus_level_df,
            save_filename=plot_save_path_specific_ind,
            locus_width_unit=0.1
        )
    else:
        print(f"\nSkipping plot for specific individual {specific_id_to_plot}: Data not found for this ID, generation, and chromosome combination.")


Starting Chromosome Ancestry Analysis: Visualization, Summary, and Block Count Plots

--- Individual IDs in Generation 'F3' ---
Total individuals in 'F3': 10
First 10 IDs (if available): [41, 42, 43, 44, 45, 46, 47, 48, 49, 50]
Last 10 IDs (if available): [41, 42, 43, 44, 45, 46, 47, 48, 49, 50]

Generating Ancestry Block Summary DataFrame...
Ancestry block summary saved to: output_data/dataframes/ancestry_block_summary.csv

First 5 rows of the Ancestry Block Summary (including new block counts):
   individual_id  diploid_chr_id generation  num_blocks_chromatid1  \
0              1               1        P_A                      1   
1              1               2        P_A                      1   
2              2               1        P_A                      1   
3              2               2        P_A                      1   
4              3               1        P_A                      1   

   num_blocks_chromatid2  
0                      1  
1                     